# Quantum Computing with Qiskit: Multiple Systems (Multi-Qubit)

This notebook extends single-qubit concepts to **systems of multiple qubits**. The central mathematical tool is the **tensor product** ($\otimes$), which combines individual qubit spaces into a larger joint Hilbert space.

Topics covered:
- Constructing multi-qubit states via tensor products of single-qubit states
- Building multi-qubit operators by tensoring single-qubit gates
- Evolving multi-qubit states with composite operators
- Creating entanglement using the CNOT (CX) gate
- Simulating partial and full measurements on multi-qubit states

In [1]:
from qiskit import __version__

print(__version__)

2.2.3


## Imports

We import `Statevector` and `Operator` from `qiskit.quantum_info` — the same classes used for single-qubit systems, now applied to multi-qubit spaces. `sqrt` is imported for normalisation constants when constructing explicit state vectors.

In [ ]:
from qiskit.quantum_info import Statevector, Operator
from numpy import sqrt

## Section 1 — Tensor Products of Quantum States

For a system of $n$ qubits, the joint state lives in a $2^n$-dimensional Hilbert space formed by the **tensor product** of each qubit's individual space:

$$\mathcal{H} = \mathcal{H}_1 \otimes \mathcal{H}_2 \otimes \cdots \otimes \mathcal{H}_n$$

`Statevector.from_label(label)` creates a named single-qubit state:
- `"0"` → $|0\rangle = \begin{pmatrix}1\\0\end{pmatrix}$, &ensp; `"1"` → $|1\rangle = \begin{pmatrix}0\\1\end{pmatrix}$

`.tensor(other)` computes $|\psi\rangle \otimes |\phi\rangle$, producing a $4$-dimensional statevector for the combined 2-qubit system. Here we form $|0\rangle \otimes |1\rangle = |01\rangle$.

In [3]:
zero = Statevector.from_label("0")
one = Statevector.from_label("1")
psi = zero.tensor(one)
display(psi.draw("latex"))

<IPython.core.display.Latex object>

### Extended Single-Qubit Basis States

Beyond the computational basis, `from_label` supports the **Hadamard (X) basis** and **Y basis**:

| Label | State | Expression |
|-------|-------|------------|
| `"+"` | $\lvert+\rangle$ | $\tfrac{1}{\sqrt{2}}\bigl(\lvert0\rangle + \lvert1\rangle\bigr)$ |
| `"-"` | $\lvert-\rangle$ | $\tfrac{1}{\sqrt{2}}\bigl(\lvert0\rangle - \lvert1\rangle\bigr)$ |
| `"r"` | $\lvert{+i}\rangle$ | $\tfrac{1}{\sqrt{2}}\bigl(\lvert0\rangle + i\lvert1\rangle\bigr)$ |
| `"l"` | $\lvert{-i}\rangle$ | $\tfrac{1}{\sqrt{2}}\bigl(\lvert0\rangle - i\lvert1\rangle\bigr)$ |

We display each state and then form the 2-qubit tensor product $\lvert+\rangle \otimes \lvert{-i}\rangle$ as `phi`.

In [10]:
plus = Statevector.from_label("+")
minus_i = Statevector.from_label("l")
plus_i = Statevector.from_label("r")
minus = Statevector.from_label("-")
phi = plus.tensor(minus_i)
display(plus.draw("latex"))
display(minus_i.draw("latex"))
display(plus_i.draw("latex"))
display(minus.draw("latex"))
display(phi.draw("latex"))

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

### The `^` Operator as Tensor Product Shorthand

Qiskit overloads `^` as a shorthand for `.tensor()`, but with **reversed argument order**: `A ^ B` computes $B \otimes A$. This follows Qiskit's qubit-ordering convention where qubit 0 is the rightmost factor in the tensor product.

So `minus ^ plus_i` is equivalent to `plus_i.tensor(minus)`, yielding $\lvert{-i}\rangle \otimes \lvert{-}\rangle$.

In [11]:
display((minus ^ plus_i).draw("latex"))

<IPython.core.display.Latex object>

## Section 2 — Tensor Products of Operators

Multi-qubit **gates** are built by tensoring single-qubit gates, producing block-structured matrices that act on each qubit independently (or jointly for entangling gates).

`Operator.from_label` creates standard single-qubit operators:
- `"H"` → Hadamard, `"I"` → Identity, `"X"` → Pauli-X (bit-flip)

`H.tensor(Id)` acts Hadamard on one qubit and Identity (no-op) on the other, giving a $4 \times 4$ matrix. `H.tensor(Id).tensor(X)` extends this to three qubits — a $8 \times 8$ matrix.

In [12]:
H = Operator.from_label("H")
Id = Operator.from_label("I")
X = Operator.from_label("X")
display(H.tensor(Id).draw("latex"))
display(H.tensor(Id).tensor(X).draw("latex"))

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

### Operator Tensor Products with the `^` Shorthand

We apply the same `^` shorthand to operators. `T ^ Id ^ X` is evaluated right-to-left by Qiskit's convention, giving $X \otimes I \otimes T$ — a 3-qubit operator displayed as an $8 \times 8$ matrix. The T gate alone is shown first for reference.

In [17]:
T = Operator.from_label("T")
display(T.draw("latex"))
display((T ^ Id ^ X).draw("latex"))

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

### Evolving a Multi-Qubit State with a Composite Operator

`Statevector.evolve(operator)` works for multi-qubit states exactly as for single qubits — the operator dimension must match the state. Here we apply `H ^ Id` (i.e. $I \otimes H$) to the 2-qubit state `phi`, transforming the first qubit with Hadamard while leaving the second unchanged.

In [18]:
display(phi.evolve(H ^ Id).draw("latex"))

<IPython.core.display.Latex object>

## Section 3 — The CNOT Gate and Entanglement

The **CNOT (Controlled-NOT)** gate acts on 2 qubits:
- **Control qubit** — unchanged
- **Target qubit** — flipped ($|0\rangle \leftrightarrow |1\rangle$) if and only if the control is $|1\rangle$

Its $4 \times 4$ matrix in the computational basis $\{|00\rangle, |01\rangle, |10\rangle, |11\rangle\}$ is defined explicitly as `CX`. Applied to $|+\rangle \otimes |0\rangle = \tfrac{1}{\sqrt{2}}(|00\rangle + |10\rangle)$, CNOT produces the **Bell state** $\tfrac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$ — a maximally entangled 2-qubit state that cannot be written as any tensor product of individual qubit states.

In [20]:
CX = Operator([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]])
psi = plus.tensor(zero)
display(CX.draw("latex"))
display(psi.evolve(CX).draw("latex"))

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

## Section 4 — Partial Measurements of Multi-Qubit States

Measuring only a **subset** of qubits in a multi-qubit system collapses those qubits to a definite outcome while leaving the remaining qubits in a renormalised residual state — this is called a **partial measurement**.

We define the 3-qubit **W state**:
$$|W\rangle = \frac{1}{\sqrt{3}}\bigl(|001\rangle + |010\rangle + |100\rangle\bigr)$$
This is an entangled state where exactly one of the three qubits is $|1\rangle$, but which one is unknown until measured.

`Statevector.measure(qargs)` accepts an optional list of qubit indices:
- `measure([0])` — measure qubit 0 only; remaining qubits stay in a 4-dimensional residual state
- `measure([0, 1])` — measure qubits 0 and 1; one qubit remains
- `measure()` — measure all qubits; the state fully collapses to a basis vector

Each call returns the outcome bit string and the post-measurement statevector.

In [23]:
w = Statevector([0, 1, 1, 0, 1, 0, 0, 0] / sqrt(3))
display(w.draw("latex"))

result, state = w.measure([0])
print(f"Measured: {result}\nState after measurement:")
display(state.draw("latex"))

result, state = w.measure([0, 1])
print(f"Measured: {result}\nState after measurement:")
display(state.draw("latex"))

result, state = w.measure()
print(f"Measured: {result}\nState after measurement:")
display(state.draw("latex"))

<IPython.core.display.Latex object>

Measured: 0
State after measurement:


<IPython.core.display.Latex object>

Measured: 01
State after measurement:


<IPython.core.display.Latex object>

Measured: 001
State after measurement:


<IPython.core.display.Latex object>